In [212]:
# Used to open and read files
from pypdf import PdfReader
# os helps us work with folders and files , eg-Finds all PDFs inside dataset folder
import os

# Import Document class to store text with metadata
from langchain_core.documents import Document

from langchain_huggingface import HuggingFaceEmbeddings

#pip install faiss-cpu langchain-community installed
from langchain_community.vectorstores import FAISS

#  Import PromptTemplate class from LangChain
# It is used to create a structured instruction that will be sent to the LLM
from langchain_core.prompts import PromptTemplate

from openai import OpenAI

from dotenv import load_dotenv
# cromr db
from groq import Groq

In [213]:
# Path to dataset folder
# .. means go one folder back from src
pdf_folder = "../dataset"

# Create a list to store all PDF pages as Document objects
documents = []


# Loop through every file inside dataset
for file in os.listdir(pdf_folder):

    # Process only pdf files
    if file.endswith(".pdf"):

        # Create full path to the pdf
        pdf_path = os.path.join(pdf_folder, file)

        # Open pdf
        reader = PdfReader(pdf_path)

        print(f"Reading: {file}")

         # enumerate gives both page number and page content
        # page_number starts from 0
        for page_number, page in enumerate(reader.pages):

            # Extract text from the current page
            text = page.extract_text()


            # Check that the page is not empty
            if text:

                # Create a Document object
                # It stores the actual text + metadata
                documents.append(

                    Document(

                        # Actual content of the PDF page
                        page_content=text,

                        # Extra information attached to the text
                        metadata={

                            # Name of the PDF file
                            "source": file,

                            # Adding 1 because page numbering starts from 1 for users
                            "page": page_number + 1
                        }
                    )
                )


print("\nAll PDFs loaded successfully with metadata!")

# Total number of pages extracted from all PDFs
print("Total documents created:", len(documents))

Reading: Clean Code Guidelines.pdf
Reading: Code_Review_Checklist.pdf
Reading: JAVA_BestPractices.pdf
Reading: OWASP_Top10.pdf

All PDFs loaded successfully with metadata!
Total documents created: 89


In [214]:
# Calculate total number of characters from all PDF pages

total_characters = 0

# Loop through each Document object
for doc in documents:
    
    # doc.page_content contains the actual text of that page
    total_characters += len(doc.page_content)


# Print the total characters extracted from all PDFs
print("Total Characters:", total_characters)

Total Characters: 150613


In [215]:
#Encoding is done here
# Open a text file in write mode with UTF-8 encoding
with open("all_documents.txt", "w", encoding="utf-8") as f:
    
    # Loop through every extracted document
    for doc in documents:
        
        # Write the PDF file name
        f.write(f"Source: {doc.metadata['source']}\n")
        
        # Write the page number
        f.write(f"Page: {doc.metadata['page']}\n")
        
        # Write the actual text content
        f.write(doc.page_content)
        
        # Add separators between documents
        f.write("\n\n" + "=" * 80 + "\n\n")

print("All documents with metadata saved to all_documents.txt")

All documents with metadata saved to all_documents.txt


In [216]:
#To check if all pdf files are read correctly
pdf_count = 0
for file in os.listdir("../dataset"):
    if file.endswith(".pdf"):
        pdf_count+=1
print(f"Number of PDF files : {pdf_count}")

Number of PDF files : 4


In [217]:
import re

# Loop through every Document object
for doc in documents:

    # Get the actual text from the document
    text = doc.page_content

    # Replace multiple spaces, tabs, and newlines with a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove tabs
    text = text.replace('\t', ' ')

    # Remove extra new lines
    text = text.replace('\n', ' ')

    # Remove multiple consecutive spaces
    text = re.sub(r' +', ' ', text)

    # Remove spaces from the beginning and end
    text = text.strip()

    # Update the cleaned text back into the document
    # Metadata (source and page) remains unchanged
    doc.page_content = text


print("Cleaning Complete!")

Cleaning Complete!


In [218]:
# Print source information of the first extracted page
print("Source:", documents[0].metadata["source"])
print("Page:", documents[0].metadata["page"])

# Print the first 2000 characters of the first document's text
print(documents[0].page_content[:2000])

Source: Clean Code Guidelines.pdf
Page: 1
Clean Code Guidelines: Dataset Specification and Code Review Comments Framework The Cognitive Economics of Clean Code For an AI code reviewer, validating functional correctness through compilation and tests is only the first layer of evaluation. 1 The true value of an automated reviewer lies in its ability to minimize the "cognitive load" of future readers. 3 In modern enterprise systems, code is read far more frequently than it is written. 4 Neglecting structural design leads to technical debt, which increases exponentially and reduces a team's capacity to ship new features. 6 During fine-tuning, the AI model must learn to quantify code complexity and suggest structural modifications that reduce it. 7 We can evaluate the structural health of a method using McCabe's Cyclomatic Complexity formula: where represents the number of control flow edges, represents the number of vertices (basic logic blocks), and represents the number of exit points or

In [219]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter object
# chunk_size = maximum number of characters in each chunk
# chunk_overlap = number of characters shared between consecutive chunks
# Overlap helps maintain context between chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# Split the Document objects into smaller chunks
# split_documents preserves the metadata (source and page number)
chunks = splitter.split_documents(documents)

# Print the total number of chunks created
print(f"Total Chunks Created: {len(chunks)}")

Total Chunks Created: 336


In [220]:
#Inspect the chunks
print(chunks[0].page_content)
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print("Total Chunks:", len(chunks))
print("Minimum Length:", min(chunk_lengths))
print("Maximum Length:", max(chunk_lengths))
print("Average Length:", sum(chunk_lengths)/len(chunk_lengths))

Clean Code Guidelines: Dataset Specification and Code Review Comments Framework The Cognitive Economics of Clean Code For an AI code reviewer, validating functional correctness through compilation and tests is only the first layer of evaluation. 1 The true value of an automated reviewer lies in its ability to minimize the "cognitive load" of future readers. 3 In modern enterprise systems, code is read far more frequently than it is written. 4 Neglecting structural design leads to technical debt,
Total Chunks: 336
Minimum Length: 99
Maximum Length: 500
Average Length: 434.51488095238096


In [221]:
#To save chunks
for i,chunk in enumerate(chunks[:5]):
    print(f"\nChunk{i+1}")
    print(chunk.page_content)


Chunk1
Clean Code Guidelines: Dataset Specification and Code Review Comments Framework The Cognitive Economics of Clean Code For an AI code reviewer, validating functional correctness through compilation and tests is only the first layer of evaluation. 1 The true value of an automated reviewer lies in its ability to minimize the "cognitive load" of future readers. 3 In modern enterprise systems, code is read far more frequently than it is written. 4 Neglecting structural design leads to technical debt,

Chunk2
far more frequently than it is written. 4 Neglecting structural design leads to technical debt, which increases exponentially and reduces a team's capacity to ship new features. 6 During fine-tuning, the AI model must learn to quantify code complexity and suggest structural modifications that reduce it. 7 We can evaluate the structural health of a method using McCabe's Cyclomatic Complexity formula: where represents the number of control flow edges, represents the number of vert

In [222]:
# Calculate total characters from all extracted PDF pages
total_characters = sum(len(doc.page_content) for doc in documents)

print("Total characters:", total_characters)

Total characters: 123359


In [223]:
#This line loads a Hugging face embedding model that will convert ur text chunks into numerical vectors
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9036.23it/s]


In [224]:
#This means the model converted your sentence into a 384-dimensional numerical vector.
vector = embedding_model.embed_query(       # embed_query()=takes text query and converts it into numerical vector
    "How can I prevent SQL Injection in Java?"
)

print("Vector length:", len(vector))

Vector length: 384


In [225]:
# You will store the embeddings of all your PDF chunks in FAISS so that when a user asks a question, your AI Code Reviewer can retrieve the most relevant coding guidelines and security rules.
#FAISS=(Facebook AI Similarity Search) is a vector database that stores embeddings and allows you to quickly find similar vectors.
vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Total vectors:", vector_db.index.ntotal)

Total vectors: 336


In [226]:
print("Total documents:", len(documents))
print("Total chunks:", len(chunks))
print("Length of first chunk:", len(chunks[0].page_content))
print("First chunk metadata:", chunks[0].metadata)

Total documents: 89
Total chunks: 336
Length of first chunk: 500
First chunk metadata: {'source': 'Clean Code Guidelines.pdf', 'page': 1}


In [227]:
#right now FAISS db is in RAM and we need to store it in disk
vector_db.save_local("../vector_db")

In [228]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.load_local(
    "../vector_db",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

In [229]:
#Test Retrieval = check whether your RAG knowledge base works.
query = "How can I prevent SQL Injection in Java?"
results = vector_db.similarity_search(
    query,
    k=3
)
print(results)

[Document(id='54a7f35e-52a3-40e0-afc4-31a79a0e304c', metadata={'source': 'OWASP_Top10.pdf', 'page': 11}, page_content='JDBC Statements that construct SQL queries dynamically, exposing the application to SQL Injection.</description> <priority>1</priority> <properties>'), Document(id='49cfffa7-d09e-445e-a7f5-20044cf4b438', metadata={'source': 'JAVA_BestPractices.pdf', 'page': 40}, page_content='binders in a PreparedStatement isolates SQL logic from execution parameters, neutralising payload executions." } ] } Works cited 1. Home · microsoft/CodeXGLUE Wiki · GitHub, accessed on June 12, 2026, https://github.com/microsoft/CodeXGLUE/wiki 2. Make your Java code smell nice and fresh - freeCodeCamp, accessed on June 12, 2026, https://www.freecodecamp.org/news/do-not-allow-bad-smells-in-your-java-code-4e8ad244393/ 3. learning-notes/books/effective-java.md at master · keyvanakbary ...,'), Document(id='cf79fcb7-8b15-42dd-8292-af6cd2406096', metadata={'source': 'OWASP_Top10.pdf', 'page': 11}, page

In [230]:
for i, doc in enumerate(results):
    print("Result", i+1)
    print(doc.page_content)
    print("-" * 50)

Result 1
JDBC Statements that construct SQL queries dynamically, exposing the application to SQL Injection.</description> <priority>1</priority> <properties>
--------------------------------------------------
Result 2
binders in a PreparedStatement isolates SQL logic from execution parameters, neutralising payload executions." } ] } Works cited 1. Home · microsoft/CodeXGLUE Wiki · GitHub, accessed on June 12, 2026, https://github.com/microsoft/CodeXGLUE/wiki 2. Make your Java code smell nice and fresh - freeCodeCamp, accessed on June 12, 2026, https://www.freecodecamp.org/news/do-not-allow-bad-smells-in-your-java-code-4e8ad244393/ 3. learning-notes/books/effective-java.md at master · keyvanakbary ...,
--------------------------------------------------
Result 3
Rule: Detect Dynamic SQL in JDBC Statements This rule flags calls to Statement.executeQuery() or Statement.executeUpdate() where the query string is constructed dynamically. 11 XML <rule name="AvoidDynamicSqlStatements" language=

In [231]:
vector_db.save_local("../vector_db")

In [232]:
results = vector_db.similarity_search(
    "How can I prevent SQL Injection in Java?",
    k=3

)

In [233]:
#A retriever is a component that takes a user query and automatically fetches most relevant chunks from FAISS
#here ,retriever also acts as a bridge between your vector database and the LLM.
retriever = vector_db.as_retriever(            #vector_db.as_retriever() = converts Faiss db into retriever object
    search_type="similarity",
    search_kwargs={"k": 3}                     #k=3 , Means return the top 3 most relevant chunks.
)

In [234]:
query = "How can I prevent SQL Injection in Java?"
docs = retriever.invoke(query)
for i, doc in enumerate(docs):
    print("Result", i + 1)
    print(doc.page_content)
    print("-" * 50)


Result 1
JDBC Statements that construct SQL queries dynamically, exposing the application to SQL Injection.</description> <priority>1</priority> <properties>
--------------------------------------------------
Result 2
binders in a PreparedStatement isolates SQL logic from execution parameters, neutralising payload executions." } ] } Works cited 1. Home · microsoft/CodeXGLUE Wiki · GitHub, accessed on June 12, 2026, https://github.com/microsoft/CodeXGLUE/wiki 2. Make your Java code smell nice and fresh - freeCodeCamp, accessed on June 12, 2026, https://www.freecodecamp.org/news/do-not-allow-bad-smells-in-your-java-code-4e8ad244393/ 3. learning-notes/books/effective-java.md at master · keyvanakbary ...,
--------------------------------------------------
Result 3
Rule: Detect Dynamic SQL in JDBC Statements This rule flags calls to Statement.executeQuery() or Statement.executeUpdate() where the query string is constructed dynamically. 11 XML <rule name="AvoidDynamicSqlStatements" language=

In [235]:
#Create a prompt object that will guide the LLM on how to answer
prompt=PromptTemplate(
    # These are placeholders that will be filled dynamically later
    # context = data retrieved from FAISS
    # question = user's query or code
    input_variables=["context", "question"],

    # The actual instruction that will be sent to the LLM
    template="""

You are an expert AI Code Reviewer specializing in Java, security, clean code, and software best practices.

Your task is to review the user's source code using only the provided reference context from OWASP, Java Best Practices, and Clean Code Guidelines.

Instructions:

1. Analyze the code for:
   - Security vulnerabilities
   - Code smells
   - Poor coding practices
   - Performance issues
   - Maintainability problems

2. For every issue found, provide:
   - Issue name
   - Severity (Low/Medium/High/Critical)
   - Explanation of why it is a problem
   - Recommended fix
   - Improved code example (if possible)
   - Reference source document

3. If the provided context does not contain enough information:
   - Clearly state: "The information is not available in the source documents."
   - Do not invent security rules or references.

Reference Context:
{context}

User Code / Question:
{question}


Generate the code review in the following format:

## Code Review Summary

### Issue 1:
- Issue Type:
- Severity:
- Description:
- Recommended Fix:
- Corrected Code:
- Source:

### Overall Recommendations:
- 
"""
)

In [236]:
# User asks a question
question = "How can I prevent SQL Injection in Java?"

# Send the question to the retriever . It converts the question into a vector and finds the most relevant chunks from FAISS
docs = retriever.invoke(question)


# Combine all retrieved chunks into a single string , doc.page_content contains the actual text of each chunk
context = "\n\n".join(
    doc.page_content for doc in docs
)


# Replace {context} and {question} placeholders
# with actual values
final_prompt = prompt.format(
    context=context,
    question=question
)


# Print the final text that will be sent to the LLM
print(final_prompt)



You are an expert AI Code Reviewer specializing in Java, security, clean code, and software best practices.

Your task is to review the user's source code using only the provided reference context from OWASP, Java Best Practices, and Clean Code Guidelines.

Instructions:

1. Analyze the code for:
   - Security vulnerabilities
   - Code smells
   - Poor coding practices
   - Performance issues
   - Maintainability problems

2. For every issue found, provide:
   - Issue name
   - Severity (Low/Medium/High/Critical)
   - Explanation of why it is a problem
   - Recommended fix
   - Improved code example (if possible)
   - Reference source document

3. If the provided context does not contain enough information:
   - Clearly state: "The information is not available in the source documents."
   - Do not invent security rules or references.

Reference Context:
JDBC Statements that construct SQL queries dynamically, exposing the application to SQL Injection.</description> <priority>1</priori